In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import platform
import re
from pathlib import Path

if platform.system() == "Darwin":   # macOS
    plt.rcParams["font.family"] = "AppleGothic"
else:                               # Windows / Linux
    plt.rcParams["font.family"] = "Malgun Gothic"

plt.rcParams["axes.unicode_minus"] = False

bus_dir_candidates = [Path("./Data/Bus"), Path("../Data/Bus")]
bus_dir = next((path for path in bus_dir_candidates if path.exists()), None)
if bus_dir is None:
    raise FileNotFoundError("버스 데이터 폴더를 찾을 수 없습니다: ./Data/Bus")

bus_files = sorted(bus_dir.glob("*.csv"))
if not bus_files:
    raise FileNotFoundError("2023년 버스 데이터 파일을 찾지 못했습니다.")

sample_df = pd.read_csv(bus_files[0], encoding="cp949", nrows=1)
time_slot_pattern = re.compile(r"(\d{1,2})시(승차|하차)총승객수")
time_slot_map = {}
for column in sample_df.columns:
    match = time_slot_pattern.fullmatch(column)
    if match:
        hour = int(match.group(1))
        traffic_type = match.group(2)
        time_slot_map.setdefault(hour, {"label": f"{hour:02d}시"})[traffic_type] = column

if not time_slot_map:
    raise ValueError("시간대 컬럼을 찾지 못했습니다.")

ordered_hours = sorted(time_slot_map)
time_slots = [time_slot_map[hour]["label"] for hour in ordered_hours]
required_columns = ["사용년월", "역명"] + [
    time_slot_map[hour][traffic_type]
    for hour in ordered_hours
    for traffic_type in ("승차", "하차")
    if traffic_type in time_slot_map[hour]
]

bus_frames = [pd.read_csv(file, encoding="cp949", usecols=required_columns) for file in bus_files]
bus_df = pd.concat(bus_frames, ignore_index=True)
bus_df["사용년월"] = bus_df["사용년월"].astype(str)
bus_2023 = bus_df[bus_df["사용년월"].str.startswith("2023")].copy()

stop_mask = bus_2023["역명"].astype(str).str.contains("여의도", na=False)
target_stops = sorted(bus_2023.loc[stop_mask, "역명"].dropna().unique())
if not target_stops:
    raise ValueError("2023년 데이터에서 여의도권 정류장을 찾지 못했습니다.")

for hour in ordered_hours:
    column_map = time_slot_map[hour]
    traffic_columns = [column_map[key] for key in ("승차", "하차") if key in column_map]
    bus_2023[column_map["label"]] = (
        bus_2023[traffic_columns]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
    )

stop_monthly = (
    bus_2023[stop_mask]
    .groupby(["사용년월", "역명"], as_index=False)[time_slots]
    .sum()
)

stop_hourly_avg = stop_monthly.groupby("역명")[time_slots].mean()

for stop_name in target_stops:
    stop_usage = stop_hourly_avg.loc[stop_name]
    fig, ax = plt.subplots(figsize=(11, 5.5))
    x_positions = range(len(time_slots))

    ax.plot(
        x_positions,
        stop_usage.values,
        marker="o",
        linewidth=2,
        markersize=5,
        color="#1f77b4"
    )
    ax.set_xticks(list(x_positions))
    ax.set_xticklabels(time_slots, rotation=45, ha="right")
    ax.set_title(f"{stop_name} 시간대별 평균 이용객 수 (2023)", fontsize=14, pad=12, wrap=True)
    ax.set_xlabel("시간대", fontsize=13)
    ax.set_ylabel("평균 이용객 수", fontsize=13)
    ax.tick_params(axis="both", labelsize=11)
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.grid(True, linestyle="--", alpha=0.4)

    fig.tight_layout()
    plt.show()
    plt.close(fig)
